In [17]:
import numpy as np
import pandas as pd
import os
import pathlib
from scipy.stats import gaussian_kde
from tqdm import tqdm
import warnings
import seaborn as sns
from skimage import util
from skimage.filters import threshold_otsu, gaussian, threshold_isodata, threshold_minimum, threshold_triangle, threshold_otsu, threshold_mean, threshold_li, try_all_threshold, threshold_yen, threshold_multiotsu
from skimage.segmentation import clear_border, watershed, expand_labels
from skimage.measure import label, regionprops, regionprops_table
from skimage.morphology import remove_small_holes, remove_small_objects, opening, ball, dilation, erosion
from skimage.transform import rescale
from skimage.feature import peak_local_max
import matplotlib.pyplot as plt
from scipy.ndimage import distance_transform_edt
from scipy import ndimage as ndi
from scipy import stats as _stats
from matplotlib.colors import to_hex
from skimage.filters import median as median_filter
from skimage.morphology import square
from tifffile import imread
import tifffile
import liffile
import napari
import colorsys
from collections import defaultdict
from skimage.morphology import closing, disk
import contextlib
import joblib
from joblib import Parallel, delayed
from PIL import ImageColor

In [18]:
# segmented = tifffile.imread(r"z:\Bel\Marina_Reconstruction\zoom3\segmented.tif")
# image_fluo= tifffile.imread(r"z:\Bel\Marina_Reconstruction\zoom3\Vimenting_SMA_VE-cadherin.lif - Ve-cad A488_SMA A594_zoom_3.tif")

segmented = tifffile.imread(r"z:\Bel\Marina_Reconstruction\zoom4\just_vasculature.tif")
image_fluo= tifffile.imread(r"z:\Bel\Marina_Reconstruction\zoom4\all_staining.tif")


fibroblasts = image_fluo[:,1,:,:]

binary = segmented > 0

# Gaussian smooth then re-threshold to smooth edges
smoothed = gaussian(binary.astype(float), sigma=1)
binary_smooth = smoothed > 0.5

# Slice-by-slice: remove small objects (area < 100) in each 2D slice
cleaned = np.zeros_like(binary_smooth, dtype=bool)
for z in range(binary_smooth.shape[0]):
    sl = binary_smooth[z]
    sl_labelled = label(sl)
    sl_cleaned = remove_small_objects(sl_labelled, min_size=1000)
    cleaned[z] = sl_cleaned > 0

# Relabel in 3D and keep only the largest component
labelled = label(cleaned)
props = pd.DataFrame(regionprops_table(labelled, properties=("label", "area")))
largest_label = int(props.loc[props["area"].idxmax(), "label"])
labelled = (labelled == largest_label).astype(np.int32) * largest_label
print(f"Kept label {largest_label}, area = {props['area'].max()} voxels")

smoothed = gaussian(labelled, sigma=3, preserve_range=True)
binary_smoothed = smoothed > threshold_otsu(smoothed)



C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_600372\2301640185.py:21: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  sl_cleaned = remove_small_objects(sl_labelled, min_size=1000)


Kept label 1, area = 4675842.0 voxels


In [19]:
viewer = napari.Viewer()
# labels_layer = viewer.add_labels(binary_smoothed.astype(np.int32), scale=(5,2.2750,2.2750))
labels_layer = viewer.add_labels(binary_smoothed.astype(np.uint32), scale = (5,1.13,1.13))
try:
    from napari.utils.colormaps import DirectLabelColormap
    labels_layer.colormap = DirectLabelColormap(color_dict={None: 'transparent', 1: 'red'})
except ImportError:
    labels_layer.color = {0: 'transparent', 1: 'magenta'}
# viewer.add_image(fibroblasts, scale=(5,2.2750,2.2750))
viewer.add_image(fibroblasts, scale=(5, 1.13,1.13))

<Image layer 'fibroblasts' at 0x21d956c0450>

In [30]:
segmented = tifffile.imread(r"Z:\Marina\Stellaris\2026.04.01_CART_setup\Bel\TEST\segmentation.tif")
image_fluo= tifffile.imread(r"Z:\Marina\Stellaris\2026.04.01_CART_setup\Bel\TEST\all_channels.tif")

# Downsample by 4 in all spatial dims
DS = 4
segmented = segmented[::DS, ::DS, ::DS]
image_fluo = image_fluo[::DS, :, ::DS, ::DS]

cart = image_fluo[:,0,:,:]

binary = segmented > 0

# Gaussian smooth then re-threshold to smooth edges
smoothed = gaussian(binary.astype(float), sigma=1)
binary_smooth = smoothed > 0.5

# Slice-by-slice: remove small objects (area < 100) in each 2D slice
cleaned = np.zeros_like(binary_smooth, dtype=bool)
for z in range(binary_smooth.shape[0]):
    sl = binary_smooth[z]
    sl_labelled = label(sl)
    sl_cleaned = remove_small_objects(sl_labelled, min_size=1000 // (DS * DS))
    cleaned[z] = sl_cleaned > 0

# Relabel in 3D and keep only the largest component
labelled = label(cleaned)
props = pd.DataFrame(regionprops_table(labelled, properties=("label", "area")))
largest_label = int(props.loc[props["area"].idxmax(), "label"])
labelled = (labelled == largest_label).astype(np.int32) * largest_label
print(f"Kept label {largest_label}, area = {props['area'].max()} voxels")

smoothed = gaussian(labelled, sigma=3, preserve_range=True)
binary_smoothed = smoothed > threshold_otsu(smoothed)
print(f"Downsampled shape: {cart.shape}")

C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_600372\1737231175.py:22: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  sl_cleaned = remove_small_objects(sl_labelled, min_size=1000 // (DS * DS))


Kept label 1, area = 915543.0 voxels
Downsampled shape: (28, 719, 718)


In [37]:
# Add 20 slices of black padding on each side in Z
Z_PAD = 20
pad_seg = np.zeros((Z_PAD, *binary_smoothed.shape[1:]), dtype=binary_smoothed.dtype)
binary_smoothed = np.concatenate([pad_seg, binary_smoothed, pad_seg], axis=0)

pad_cart = np.zeros((Z_PAD, *cart.shape[1:]), dtype=cart.dtype)
cart = np.concatenate([pad_cart, cart, pad_cart], axis=0)

print(f"Padded shapes: cart={cart.shape}, segmentation={binary_smoothed.shape}")


Padded shapes: cart=(68, 719, 718), segmentation=(68, 719, 718)


In [38]:
# --- Ellipsoid at intensity-weighted COM of CART channel ---
# Adjust these radii (in voxels) by trial and error
radius_z = 30    # half-length along Z
radius_y = 90    # half-length along Y
radius_x = 90    # half-length along X

# Compute intensity-weighted centre of mass
com = ndi.center_of_mass(cart)
cz, cy, cx = com
cz -= 2
cx += 10
cy += 10
print(f"CART intensity COM: z={cz:.1f}, y={cy:.1f}, x={cx:.1f}")

# Build coordinate grids and evaluate ellipsoid equation
zz, yy, xx = np.ogrid[:cart.shape[0], :cart.shape[1], :cart.shape[2]]
ellipsoid_mask = (((zz - cz) / radius_z) ** 2 +
                  ((yy - cy) / radius_y) ** 2 +
                  ((xx - cx) / radius_x) ** 2) <= 1.0

cart_ellipsoid = ellipsoid_mask.astype(np.int32)
print(f"Ellipsoid voxels: {cart_ellipsoid.sum()}")


CART intensity COM: z=29.8, y=363.9, x=364.4
Ellipsoid voxels: 1017861


In [39]:
viewer = napari.Viewer()
labels_layer = viewer.add_labels(cart_ellipsoid.astype(np.int32), scale=(2,1.1018,1.1018))
try:
    from napari.utils.colormaps import DirectLabelColormap
    labels_layer.colormap = DirectLabelColormap(color_dict={None: 'transparent', 1: 'gray'})
except ImportError:
    labels_layer.color = {0: 'transparent', 1: 'magenta'}
labels_layer = viewer.add_labels(binary_smoothed.astype(np.int32), scale=(2,1.1018,1.1018))
try:
    from napari.utils.colormaps import DirectLabelColormap
    labels_layer.colormap = DirectLabelColormap(color_dict={None: 'transparent', 1: 'red'})
except ImportError:
    labels_layer.color = {0: 'transparent', 1: 'darkorange'}
viewer.add_image(cart, scale=(2,1.1018,1.1018) )

<Image layer 'cart' at 0x21db949c750>